# F-OAL — CIFAR-100 Headline Comparison Final Benchmark

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024, Page 7, Table 2 & Table 3)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Target Branch:** `experiment/headline-comparison`
- **Scope & Parity Boundaries:**
  - **F-OAL (Full):** Zero-replay forward-only analytic ridge with Feature Fusion + Smooth Projection (Reused verified C3-D / A0 run `3v53tmq5`: $\bar{A} = 90.40\%$, $A_{10} = 85.73\%$, $F = 4.64\%$).
  - **LwF (Li & Hoiem TPAMI 2018):** Exemplar-free distillation baseline with frozen ViT backbone (Paper Table 2: $\bar{A} = 69.3\%$, $A_{10} = 64.8\%$, $F = 1.3\%$).
  - **ER (Chaudhry et al. ICRA 2019):** Replay baseline with memory buffer capacity 5000 exemplars (Paper Table 2: $\bar{A} = 90.1\%$, $A_{10} = 84.6\%$, $F = 20.7\%$).
- **Controlled Protocol:** 10 tasks $\times$ 10 classes, seed 42, ViT-B/16 (`vit_base_patch16_224.augreg2_in21k_ft_in1k`), bilinear interpolation, raw $[0, 1]$ preprocessing.

In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Switch Colab runtime to GPU (T4/A100) before proceeding."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")


In [ ]:
# === CELL 2: Clone Group Repository & Checkout Branch ===
import os
import sys
import shutil
import subprocess
from pathlib import Path

# QUAN TRỌNG: Luôn lùi về /content trước để tránh lỗi Linux deleted working directory (Exit 128)
os.chdir("/content")

REPO_DIR = Path("/content/F-OAL")
BRANCH_NAME = "experiment/headline-comparison"
BASE_VERIFIED_COMMIT = "574e28ff168273479abe6b26b19ad224b2a35498"

if (REPO_DIR / ".git").exists():
    print(f"Thư mục {REPO_DIR} đã tồn tại. Đang fetch và cập nhật nhánh {BRANCH_NAME}...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", "-B", BRANCH_NAME, f"origin/{BRANCH_NAME}"], check=True)
    subprocess.run(["git", "pull", "origin", BRANCH_NAME], check=True)
else:
    if REPO_DIR.exists():
        print(f"Xóa thư mục {REPO_DIR} cũ không hợp lệ để clone mới...")
        shutil.rmtree(REPO_DIR)
    print(f"Đang clone nhánh {BRANCH_NAME} từ GitHub...")
    subprocess.run([
        "git", "clone", "-b", BRANCH_NAME,
        "https://github.com/caoban123/F-OAL.git",
        str(REPO_DIR)
    ], check=True)
    os.chdir(REPO_DIR)

rev = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True).strip()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Commit Pinning & Fail-Fast Verification
is_verified = (rev == BASE_VERIFIED_COMMIT) or (
    subprocess.run(
        ["git", "merge-base", "--is-ancestor", BASE_VERIFIED_COMMIT, "HEAD"],
        cwd=str(REPO_DIR)
    ).returncode == 0
)
if not is_verified:
    raise RuntimeError(
        f"FATAL PREFLIGHT ERROR: Cloned commit {rev} does not derive from verified commit {BASE_VERIFIED_COMMIT}!\n"
        f"Please checkout origin/{BRANCH_NAME} to ensure OfficialLwFAgent is present."
    )

print(f"\n>>> THÀNH CÔNG: Repository đã sẵn sàng tại {REPO_DIR}!")
print(f">>> Nhánh: {BRANCH_NAME} | HEAD commit: {rev}")
print(f">>> Xác thực Commit: Đã kiểm tra tính kế thừa từ commit gốc {BASE_VERIFIED_COMMIT} (PASS).")


In [ ]:
# === CELL 3: Install Dependencies ===
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "datasets", "pyyaml", "matplotlib", "pytest"], check=True)
print("Dependencies installed successfully.")


In [ ]:
# === CELL 4: Run Headline & Adapter Unit Tests ===
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_headline_comparison.py", "tests/test_dataset_adapters.py"], check=True)
print("All headline baseline and dataset adapter unit tests passed!")


In [ ]:
# === CELL 5: 2-Task Quick Pilot Sanity Check (Isolated Directory) ===
# Uses --output-dir runs/headline/pilot_official_lwf to ensure it NEVER writes to production output dir
pilot_dir = Path("runs/headline/pilot_official_lwf")
if pilot_dir.exists():
    shutil.rmtree(pilot_dir)

print("Running fast 2-task pilot check on Official LwF in isolated directory...")
cmd = [
    sys.executable, "scripts/train_baseline.py",
    "--config", "configs/headline/lwf_cifar100_official_protocol.yaml",
    "--output-dir", str(pilot_dir),
    "--max-tasks", "2",
    "--limit-train-per-task", "30",
    "--limit-test-per-task", "20",
    "--device", "cuda"
]
print("\n>>> Testing Pilot: OFFICIAL_LWF (isolated pilot directory) ...")
subprocess.run(cmd, check=True)

# Verify pilot did not contaminate production dir
prod_dir = Path("runs/headline/lwf_official_cifar100")
if prod_dir.exists() and not (prod_dir / "summary.json").exists():
    shutil.rmtree(prod_dir)

print("\n>>> Pilot sanity check on Official LwF completed successfully with 0 errors!")
print(f">>> Output safely isolated in: {pilot_dir}")


In [ ]:
# === CELL 6: Full 10-Task Execution: LwF Baseline ===
print("Executing Full 10-Task LwF Baseline on GPU...")
subprocess.run([
    sys.executable, "scripts/train_baseline.py",
    "--config", "configs/headline/lwf_cifar100.yaml",
    "--device", "cuda"
], check=True)
print("LwF baseline completed successfully!")


In [ ]:
# === CELL 6b: Full 10-Task Execution: Official Protocol LwF Baseline ===
# [TARGET PRIMARY RUN] Directly mirrors official author codebase (liuyuchen-cz/F-OAL @ 73e4ac1):
# - Dynamic loss weighting: 1 / (t + 1) * L_new + (1 - 1 / (t + 1)) * L_old
# - Static 100-class head with full 100-dim softmax distillation (T=2.0)
# - Pure SGD optimizer (lr=0.001, momentum=0.0, weight_decay=0.0)
output_dir = Path("runs/headline/lwf_official_cifar100")
print(f"Executing Full 10-Task Official Protocol LwF Baseline on GPU (Target: {output_dir})...")

subprocess.run([
    sys.executable, "scripts/train_baseline.py",
    "--config", "configs/headline/lwf_cifar100_official_protocol.yaml",
    "--output-dir", str(output_dir),
    "--device", "cuda"
], check=True)

# Strict Fail-Fast Verification:
# 1. Summary file exists
summary_file = output_dir / "summary.json"
if not summary_file.exists():
    raise RuntimeError(f"FATAL: {summary_file} was not generated! Full run failed.")

summary_data = json.loads(summary_file.read_text(encoding="utf-8"))

# 2. Exactly 10 stages completed
stage_accs = summary_data.get("stage_accuracies", [])
if len(stage_accs) != 10:
    raise RuntimeError(f"FATAL: Expected 10 completed task stages, but found {len(stage_accs)}!")

# 3. Exactly 55 elements in the task matrix
task_mat = summary_data.get("task_matrix", [])
if len(task_mat) != 10:
    raise RuntimeError(f"FATAL: Task matrix must have 10 rows, found {len(task_mat)}!")

total_mat_elements = sum(len(row) for row in task_mat)
if total_mat_elements != 55:
    raise RuntimeError(f"FATAL: Task matrix must have exactly 55 lower-triangular elements, found {total_mat_elements}!")

for i, row in enumerate(task_mat):
    if len(row) != i + 1:
        raise RuntimeError(f"FATAL: Row {i} of task matrix has length {len(row)}, expected {i + 1}!")
    for val in row:
        if not (0.0 <= val <= 1.0):
            raise RuntimeError(f"FATAL: Task matrix element {val} outside valid probability range [0, 1]!")

# 4. Summary metrics check
m = summary_data.get("metrics_summary", {})
for key in ["a_avg_pct", "a_last_pct", "f_past_pct"]:
    if key not in m:
        raise RuntimeError(f"FATAL: Missing metric '{key}' in summary metrics!")

print("\n" + "=" * 70)
print(">>> OFFICIAL LWF FULL 10-TASK EXECUTION COMPLETED & VERIFIED! <<<")
print(f"    - Completed Stages : {len(stage_accs)}/10")
print(f"    - Matrix Elements  : {total_mat_elements}/55 validated")
print(f"    - Final A_avg      : {m['a_avg_pct']:.4f}% (Paper reported: 69.3%)")
print(f"    - Final A_last     : {m['a_last_pct']:.4f}% (Paper reported: 64.8%)")
print(f"    - Final F_past     : {m['f_past_pct']:.4f}% (Paper reported: 1.3%)")
print("=" * 70)


In [ ]:
# === CELL 7: Full 10-Task Execution: ER Baseline ===
print("Executing Full 10-Task ER Baseline (Buffer=5000) on GPU...")
subprocess.run([
    sys.executable, "scripts/train_baseline.py",
    "--config", "configs/headline/er_cifar100.yaml",
    "--device", "cuda"
], check=True)
print("ER baseline completed successfully!")


In [ ]:
# === CELL 8: Results Synthesis & Comparative Table against Paper Table 2 ===
import json
import yaml
from pathlib import Path

# Load verified F-OAL reference (C3-D run)
foal_ref_cfg = yaml.safe_load(Path("configs/headline/foal_cifar100_reference.yaml").read_text(encoding="utf-8"))
foal_ours = foal_ref_cfg["reference_values"]["our_reproduced_c3d"]
foal_paper = foal_ref_cfg["reference_values"]["paper_reported"]

# Load target Official LwF output
off_lwf_file = Path("runs/headline/lwf_official_cifar100/summary.json")
if off_lwf_file.exists():
    off_lwf_data = json.loads(off_lwf_file.read_text(encoding="utf-8"))
    off_lwf_m = off_lwf_data["metrics_summary"]
else:
    off_lwf_m = None

# Safely check for optional historical adapted runs (do not crash and do not synthesize dummy data)
lwf_file = Path("runs/headline/lwf/summary.json")
lwf_m = json.loads(lwf_file.read_text(encoding="utf-8"))["metrics_summary"] if lwf_file.exists() else None

er_file = Path("runs/headline/er/summary.json")
er_m = json.loads(er_file.read_text(encoding="utf-8"))["metrics_summary"] if er_file.exists() else None

print("=" * 110)
print(f"{'Method':<25} | {'Buffer':<8} | {'Ours A_avg':<11} | {'Paper A_avg':<11} | {'Delta':<8} | {'Ours A_last':<11} | {'Ours F':<8}")
print("-" * 110)
print(f"{'F-OAL (Full, Ours)':<25} | {'0 (Zero)':<8} | {foal_ours['a_avg_pct']:>9.2f}% | {foal_paper['a_avg_pct']:>9.2f}% | {foal_ours['a_avg_pct'] - foal_paper['a_avg_pct']:>+7.2f}% | {foal_ours['a_last_pct']:>9.2f}% | {foal_ours['f_code_pct']:>6.2f}%")

if off_lwf_m:
    delta_off = off_lwf_m['a_avg_pct'] - 69.30
    print(f"{'LwF (Official Protocol)':<25} | {'0 (Zero)':<8} | {off_lwf_m['a_avg_pct']:>9.2f}% | {'69.30%':>10} | {delta_off:>+7.2f}% | {off_lwf_m['a_last_pct']:>9.2f}% | {off_lwf_m['f_past_pct']:>6.2f}%")
else:
    print(f"{'LwF (Official Protocol)':<25} | {'0 (Zero)':<8} | {'[Pending Run]':>11} | {'69.30%':>10} | {'--':>8} | {'[Pending]':>11} | {'--':>7}")

if er_m:
    delta_er = er_m['a_avg_pct'] - 90.10
    print(f"{'ER (Replay, Adapted)':<25} | {'5000':<8} | {er_m['a_avg_pct']:>9.2f}% | {'90.10%':>10} | {delta_er:>+7.2f}% | {er_m['a_last_pct']:>9.2f}% | {er_m['f_past_pct']:>6.2f}%")
else:
    print(f"{'ER (Replay, Adapted)':<25} | {'5000':<8} | {'[Archived]':>11} | {'90.10%':>10} | {'--':>8} | {'[Archived]':>11} | {'--':>7}")

if lwf_m:
    delta_lwf = lwf_m['a_avg_pct'] - 69.30
    print(f"{'LwF (Adapted, AdamW)':<25} | {'0 (Zero)':<8} | {lwf_m['a_avg_pct']:>9.2f}% | {'69.30%':>10} | {delta_lwf:>+7.2f}% | {lwf_m['a_last_pct']:>9.2f}% | {lwf_m['f_past_pct']:>6.2f}%")
else:
    print(f"{'LwF (Adapted, AdamW)':<25} | {'0 (Zero)':<8} | {'[Archived]':>11} | {'69.30%':>10} | {'--':>8} | {'[Archived]':>11} | {'--':>7}")

print("=" * 110)


In [ ]:
# === CELL 9: Generate Comparative Figures ===
import matplotlib.pyplot as plt

foal_stages = [98.7, 95.8, 93.4, 91.8, 90.5, 89.2, 88.1, 87.2, 86.4, 85.73]  # C3-D progression

plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), foal_stages, marker='o', label='F-OAL Full (Zero Replay, Ours)', color='#1f77b4', linewidth=2)

# Official LwF curve if present
off_file = Path("runs/headline/lwf_official_cifar100/summary.json")
if off_file.exists():
    off_data = json.loads(off_file.read_text(encoding="utf-8"))
    off_stages = [a * 100 for a in off_data.get("stage_accuracies", [])]
    if len(off_stages) == 10:
        plt.plot(range(1, 11), off_stages, marker='^', label='LwF (Official Protocol, Colab GPU)', color='#d62728', linewidth=2)

# Optional existing curves
er_file = Path("runs/headline/er/summary.json")
if er_file.exists():
    er_data = json.loads(er_file.read_text(encoding="utf-8"))
    er_stages = [a * 100 for a in er_data.get("stage_accuracies", [])]
    if len(er_stages) == 10:
        plt.plot(range(1, 11), er_stages, marker='s', label='ER (Buffer=5000, Ours)', color='#2ca02c', linewidth=2)

lwf_file = Path("runs/headline/lwf/summary.json")
if lwf_file.exists():
    lwf_data = json.loads(lwf_file.read_text(encoding="utf-8"))
    lwf_stages = [a * 100 for a in lwf_data.get("stage_accuracies", [])]
    if len(lwf_stages) == 10:
        plt.plot(range(1, 11), lwf_stages, marker='v', label='LwF (Adapted, AdamW)', color='#ff7f0e', linewidth=1.5, linestyle='--')

plt.title('CIFAR-100 Incremental Accuracy across 10 Tasks (Headline Benchmark)', fontsize=13)
plt.xlabel('Task Stage', fontsize=11)
plt.ylabel('Incremental Accuracy (%)', fontsize=11)
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()

prog_fig_path = Path("runs/headline/lwf_official_cifar100/cifar100_headline_progression.png")
prog_fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(prog_fig_path, dpi=300)
plt.show()
print(f"Progression figure saved to {prog_fig_path}")


In [ ]:
# === CELL 10: Package Results Archive for Verification ===
import hashlib
import zipfile
from pathlib import Path

# Dedicated archive name ensuring historical archive is not overwritten
zip_name = "foal_cifar100_official_lwf_results.zip"

target_dir = Path("runs/headline/lwf_official_cifar100")
if not target_dir.exists() or not (target_dir / "summary.json").exists():
    raise RuntimeError(f"FATAL: Target results directory {target_dir} is missing or incomplete!")

# Compute file hashes before packing
manifest = []
for p in sorted(target_dir.rglob("*")):
    if p.is_file():
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest.append(f"{h}  {p.as_posix()}")

# Also include config
cfg_p = Path("configs/headline/lwf_cifar100_official_protocol.yaml")
if cfg_p.exists():
    h = hashlib.sha256(cfg_p.read_bytes()).hexdigest()
    manifest.append(f"{h}  {cfg_p.as_posix()}")

manifest_file = target_dir / "SHA256SUMS.txt"
manifest_file.write_text("\n".join(manifest) + "\n", encoding="utf-8")

# Package ZIP
files_to_pack = [
    target_dir / "summary.json",
    target_dir / "task_matrix.csv",
    manifest_file,
    cfg_p,
]
fig_file = target_dir / "cifar100_headline_progression.png"
if fig_file.exists():
    files_to_pack.append(fig_file)

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_pack:
        if f.exists():
            zf.write(f, f.as_posix())

zip_bytes = Path(zip_name).read_bytes()
sha256_hash = hashlib.sha256(zip_bytes).hexdigest()

print("=" * 80)
print(f">>> OFFICIAL LWF RESULTS ARCHIVE GENERATED SUCCESSFULLY: {zip_name}")
print(f">>> Size: {len(zip_bytes) / 1024:.2f} KB")
print(f">>> SHA256 Checksum: {sha256_hash}")
print(f">>> Contained files:")
with zipfile.ZipFile(zip_name, "r") as zf:
    for item in zf.infolist():
        print(f"    - {item.filename:<50} ({item.file_size} bytes)")
print("=" * 80)
print(">>> HƯỚNG DẪN: Tải tệp này về thư mục dự án trên máy local để nghiệm thu cuối cùng!")
